# 03 — Periodic Phase Encoding

노트북은 Step 3 실험 실행만 담당하고, 공통 학습/평가 로직은 `src/`로 분리했습니다.


## 1. Setup

In [ ]:
import importlib.util
import sys
from pathlib import Path

if importlib.util.find_spec('google') is not None and importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive')

REPO_ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next((p for p in REPO_ROOT_CANDIDATES if (p / 'src' / 'paths.py').exists()), None)
assert REPO_ROOT is not None, 'repo root with src/paths.py not found; run this notebook from the cloned repository'
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from paths import ARTIFACT_ROOT, DATA_DIR, CHECKPOINTS_DIR, FIGURES_DIR, ensure_artifact_dirs
from reproducibility import describe_project_paths

ensure_artifact_dirs()
describe_project_paths(REPO_ROOT, SRC_DIR, ARTIFACT_ROOT)


## 2. Dependencies

In [ ]:
!pip install -q -r {REPO_ROOT / 'requirements.txt'}
print('✓ requirements.txt dependencies are ready')


## 3. Imports + Config

In [ ]:
import gymnasium as gym
import torch

from configs import get_experiment_config
from experiment_plots import (
    plot_action_chunks,
    plot_frequency_sweep,
    plot_loss_curve,
    plot_sample_histogram,
    print_step3_vs_step4_table,
)
from experiment_runner import (
    apply_best_ema_after_training,
    build_model_with_sanity_check,
    build_noise_scheduler,
    load_data_and_build_loaders,
    rollout_at_frequency,
    run_frequency_sweep,
    sample_frequency_variants,
    sample_single_batch,
    train_or_load_checkpoint,
)
from phase import (
    controllability_sweep_frequencies,
    legacy_periodic_sweep_frequencies,
    periodic_offline_frequencies,
    trajectory_offline_frequencies,
    training_frequency_triplet,
)
from reproducibility import resolve_device
from sampling import diagnose_obs_distribution

device = resolve_device()
print(f'PyTorch {torch.__version__}, device={device}')

cfg = get_experiment_config('periodic_phase')
train_cond_fn = cfg.resolve_train_cond_fn()
sample_cond_fn = cfg.resolve_sample_cond_fn()
print(f'Experiment config: {cfg.name} — {cfg.display_name}')


## 4. Data

In [ ]:
data, train_ds, val_ds, train_loader, val_loader = load_data_and_build_loaders(cfg, DATA_DIR)
f_mean, f_min, f_max = training_frequency_triplet(data)


## 5. Model + Scheduler

In [ ]:
model = build_model_with_sanity_check(cfg, data, device=device, train_loader=train_loader)
noise_scheduler, ns_config, NUM_INFERENCE_STEPS = build_noise_scheduler(cfg)
ema = cfg.build_ema(model)


## 6. Train or Load

In [ ]:
TRAIN = True
train_losses, val_log, best_ema_state, CKPT_PATH = train_or_load_checkpoint(
    train=TRAIN, cfg=cfg, model=model, ema=ema, noise_scheduler=noise_scheduler,
    train_loader=train_loader, val_loader=val_loader, checkpoints_dir=CHECKPOINTS_DIR, device=device,
)
apply_best_ema_after_training(TRAIN, best_ema_state, ema)


## 7. Loss Curve

In [ ]:
plot_loss_curve(train_losses, val_log, FIGURES_DIR / cfg.artifacts.loss_plot_name, title=cfg.display_name)


## 8. Offline Phase Sensitivity

In [ ]:
freqs, labels = periodic_offline_frequencies(data)
ep_obs = val_ds[0]['obs'].unsqueeze(0)
samples_by_freq = sample_frequency_variants(
    model, ema, ns_config, ep_obs, data, sample_cond_fn, freqs, labels,
    device=device, num_inference_steps=NUM_INFERENCE_STEPS, dt=cfg.evaluation.dt, seed=data['seed'],
)
plot_action_chunks(
    samples_by_freq, FIGURES_DIR / 'phase_periodic_freq_sweep.png',
    title='Periodic DP — same obs, different rollout phase frequencies', act_dim=data['ACT_DIM'],
)


## 9. Ant Rollout — In Distribution

In [ ]:
env = gym.make('Ant-v5')
results_in = rollout_at_frequency(
    model, ema, env, ns_config, data, sample_cond_fn,
    freq_hz=f_mean, n_seeds=cfg.evaluation.n_seeds, max_steps=1000,
    num_inference_steps=NUM_INFERENCE_STEPS, deterministic_sampling=cfg.evaluation.deterministic_sampling,
    device=device, dt=cfg.evaluation.dt,
)


## 10. Controllability Sweep

In [ ]:
sweep_freqs = legacy_periodic_sweep_frequencies(data)
sweep_results = run_frequency_sweep(
    model, ema, env, ns_config, data, sample_cond_fn, sweep_freqs,
    n_seeds=3, max_steps=1000, num_inference_steps=NUM_INFERENCE_STEPS,
    deterministic_sampling=cfg.evaluation.deterministic_sampling, device=device, dt=cfg.evaluation.dt,
)
plot_frequency_sweep(
    sweep_results, data, FIGURES_DIR / 'phase_periodic_controllability.png',
    title_prefix='Periodic DP', max_steps=1000,
)


## 11. Vanilla vs Periodic Note

In [ ]:
print('Vanilla/Periodic 정밀 비교는 동일 protocol을 사용하는 05_evaluation.ipynb에서 수행하세요.')
